# Mass budget

A worked example of the `quicksat` mass budget against the sample satellite in `data/`: a small Earth observation platform with a telescope payload, hydrazine propulsion, and a separation interface split between the satellite and the launcher.

In [1]:
# Change log. LAST_CHANGE and CHANGE_NOTE are typed in by hand: update them whenever
# the inputs move -- an equipment item swapped for an alternative, a margin lowered now
# that CDR is passed -- so that the figures below can be read against what produced them.
# The run time is recorded automatically, and says how stale the outputs stored in this
# notebook are relative to that last change.
from datetime import datetime

LAST_CHANGE = "2026-09-15"
CHANGE_NOTE = "Initial sample: EO platform, telescope payload, hydrazine propulsion."

print(f"last change  {LAST_CHANGE}")
print(f"             {CHANGE_NOTE}")
print(f"last run     {datetime.now().astimezone():%Y-%m-%d %H:%M %Z}")

last change  2026-09-15
             Initial sample: EO platform, telescope payload, hydrazine propulsion.
last run     2026-09-21 00:44 CEST


In [2]:
import os
from pathlib import Path

import pandas as pd

# Make the in-development quicksat package importable without installing it: walk up
# from the current directory to the repo root (the folder that holds the quicksat
# package) and switch to it. Works whether the notebook runs from sample/, the repo
# root, or the docs build.
here = Path.cwd()
repo_root = next(
    (p for p in (here, *here.parents) if (p / "quicksat" / "__init__.py").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError("could not locate the quicksat repo root")
os.chdir(repo_root)

from quicksat.mass.budget import MassBudget

pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## Loading

The satellite is described by two files: a flat equipment CSV, and a config YAML holding the system margins and harness fractions, keyed on location.

In [3]:
DATA = Path("sample") / "data"
mass_data = MassBudget.from_csv(DATA / "equipment.csv", DATA / "mass_budget_config.yaml")

f"Loaded mass data file and config: {len(mass_data.eqpt_table)} rows, harness included"

'Loaded mass data file and config: 33 rows, harness included'

## The equipment table

Everything the budget knows, in one table. Nothing is nested: `location` is the computation axis — it drives the system margin, the harness fraction, and what survives separation — while `responsibility` and `subsystem` exist so the same rows can be summed different ways.

Two things in here were not typed into the CSV. Masses were parsed with pint on load, so the IMU's `750 g` has already become kilograms. And the two `harness_*` rows at the bottom were derived from each location's equipment mass, then given their own contingency.

In [4]:
mass_data.eqpt_table.sort_values("location")

,equipment_id,equipment_name,location,responsibility,subsystem,eqpt_mass,eqpt_margin,number_of_units,mass_class,comments,eqpt_total_mass
29,sep_ring_lv,"Adapter ring, launcher side",Launcher,Launcher,Structure,11.00,10.00,1,equipment,Stays with the launcher,11.00
30,clampband,Clampband and pyros,Launcher,Launcher,Structure,4.20,10.00,1,equipment,Stays with the launcher,4.20
31,harness_payload,Payload harness,Payload,Payload,Harness,5.20,25.00,1,equipment,Derived: 5.0% of 104.000 kg equipment mass,5.20
7,star_tracker,Jena Astro HP,Payload,Platform,ADCS,1.20,5.00,2,equipment,Redundant pair,2.40
6,heaters,Heater lines and thermistors,Payload,Payload,Thermal,1.20,15.00,1,equipment,,1.20
0,telescope,TMA telescope assembly,Payload,Payload,Instrument,62.00,20.00,1,equipment,,62.00
4,optical_bench,Optical bench,Payload,Payload,Structure,12.00,20.00,1,equipment,,12.00
3,payload_radiator,Payload radiator,Payload,Payload,Thermal,3.40,20.00,1,equipment,,3.40
2,payload_electronics,Video processing unit,Payload,Payload,OBDH,8.00,10.00,1,equipment,,8.00
1,focal_plane,Focal plane assembly,Payload,Payload,Instrument,11.50,15.00,1,equipment,,11.50


## The budget as a table

`tabulated_mass()` renders the whole budget as a table: every item grouped into
subsystem blocks within each location, then a subtotal before the system margin, and
a merged location-total row that folds in the system margin. The location totals are
followed by the dry mass, the propellant, and the wet mass.

Propellant appears once, at the bottom, and is held out of the blocks above it — so
every subtotal on the way down is a dry mass and the column adds up as it reads.

The mass columns trace the progression at each level. For equipment rows, `Mass [kg]`
is the mass of one unit, `Total Mass [kg]` is the pre-margin total for all units, and
`Total + Margin [kg]` adds the per-item contingency. On the location-total row the
same columns fold in the system margin: `Total Mass [kg]` carries the
equipment-margined subtotal, `Margin [%]` the system margin rate, and
`Total + Margin [kg]` the final figure after both.

What comes back is a pandas `Styler` rather than a plain frame, so cells that do not
apply to a row come out blank instead of `NaN`. The numbers are still there as `.data`
if you want to compute with them.

In [5]:
mass_data.tabulated_mass(in_orbit=True)

Item,Name,Subsystem,Units,Mass [kg],Total Mass [kg],Margin [%],Total + Margin [kg]
telescope,TMA telescope assembly,Instrument,1,62.00,62.00,20,74.40
focal_plane,Focal plane assembly,Instrument,1,11.50,11.50,15,13.22
payload_electronics,Video processing unit,OBDH,1,8.00,8.00,10,8.80
payload_radiator,Payload radiator,Thermal,1,3.40,3.40,20,4.08
mli,Multi-layer insulation,Thermal,1,3.50,3.50,20,4.20
heaters,Heater lines and thermistors,Thermal,1,1.20,1.20,15,1.38
optical_bench,Optical bench,Structure,1,12.00,12.00,20,14.40
star_tracker,Jena Astro HP,ADCS,2,1.20,2.40,5,2.52
harness_payload,Payload harness,Harness,1,5.20,5.20,25,6.50
,Payload subtotal (before system margin),,,,109.20,,129.50


Passing `in_orbit=False` puts the whole table before separation instead: a Launcher block joins the others, and the totals become the on-ground masses.

Items are grouped by subsystem within each location, but the per-subsystem subtotal lines are off by default — on a table this size they crowd out the items. `subsystem_subtotals=True` adds them back.

## Extracting single values

Every query takes the same four flags, all defaulting to `True`, so the usual question is a bare call and each deviation is one explicit switch: `propellant` is the percentage of the propellant load to count, `sys_margin` applies the location's system margin, `eqpt_margin` applies the per-item margin, and `in_orbit` drops hardware left with the launcher.

Two of those flags give the four masses normally quoted for a satellite.

In [6]:
print(f"launch mass (on ground, wet)   {mass_data.on_ground_mass():~.2f}")
print(f"dry mass at launch             {mass_data.on_ground_mass(propellant=0):~.2f}")
print(f"separated wet mass (in orbit)  {mass_data.in_orbit_mass():~.2f}")
print(f"in-orbit dry mass              {mass_data.in_orbit_mass(propellant=0):~.2f}")
print()
print("propellant is a percentage, not a switch, so any point in the mission works:")
for pct in (100, 75, 50, 25, 0):
    print(f"  {pct:3d}% propellant remaining   {mass_data.in_orbit_mass(propellant=pct):~.2f}")
print()
print(f"difference on ground vs orbit  {(mass_data.on_ground_mass() - mass_data.in_orbit_mass()):~.2f}")
print("  = the adapter ring half and clampband that stay with the launcher")

launch mass (on ground, wet)   487.34 kg
dry mass at launch             465.34 kg
separated wet mass (in orbit)  470.62 kg
in-orbit dry mass              448.62 kg

propellant is a percentage, not a switch, so any point in the mission works:
  100% propellant remaining   470.62 kg
   75% propellant remaining   465.12 kg
   50% propellant remaining   459.62 kg
   25% propellant remaining   454.12 kg
    0% propellant remaining   448.62 kg

difference on ground vs orbit  16.72 kg
  = the adapter ring half and clampband that stay with the launcher


### By platform and payload

`platform_mass` and `payload_mass` take a `by_responsibility` switch, because Platform
and Payload name both a location *and* a responsibility. In this satellite the star
tracker sits physically on the payload but belongs to the platform team, so the two
axes disagree by exactly its margined mass.

Note these count the full propellant load by default, so the platform figure carries
the hydrazine.

In [7]:
print(f"platform, by location        {mass_data.platform_mass():~.2f}")
print(f"platform, by responsibility  {mass_data.platform_mass(by_responsibility=True):~.2f}")
print()
print(f"payload,  by location        {mass_data.payload_mass():~.2f}")
print(f"payload,  by responsibility  {mass_data.payload_mass(by_responsibility=True):~.2f}")
print()
print(f"platform, dry                {mass_data.platform_mass(propellant=0):~.2f}")
print(f"propellant                   {mass_data.propellant_mass():~.2f}")

platform, by location        315.21 kg
platform, by responsibility  318.24 kg

payload,  by location        155.41 kg
payload,  by responsibility  152.38 kg

platform, dry                293.21 kg
propellant                   22.00 kg


### By subsystem

`subsystem_mass` carries no system margin — margins of that kind are held at the platform and payload level and cannot be attributed to a subsystem. It needs no `propellant` flag either: the propellant row names Propulsion as its subsystem, so a propulsion query picks the full load up on its own.

In [8]:
for subsystem in ["ADCS", "EPS", "Structure", "Propulsion"]:
    with_margin = mass_data.subsystem_mass(subsystem)
    raw = mass_data.subsystem_mass(subsystem, eqpt_margin=False)
    print(f"{subsystem:12s} {raw:~8.2f} raw  ->  {with_margin:~8.2f} with margin")

ADCS            36.05 kg raw  ->     39.50 kg with margin
EPS             45.30 kg raw  ->     52.33 kg with margin
Structure       95.00 kg raw  ->    113.10 kg with margin
Propulsion      32.20 kg raw  ->     33.66 kg with margin


The grouped view gives the same thing for every subsystem at once. Because the margin flags apply here too, the three margin layers are the same call three times.

In [9]:
pd.DataFrame(
    {
        "no_margins": mass_data.by_subsystem(eqpt_margin=False, sys_margin=False)["mass"],
        "with_eqpt_margin": mass_data.by_subsystem(sys_margin=False)["mass"],
        "with_sys_margin": mass_data.by_subsystem()["mass"],
    }
)

,no_margins,with_eqpt_margin,with_sys_margin
subsystem,,,
ADCS,36.05,39.50,47.40
COMM,11.10,12.30,14.77
EPS,45.30,52.34,62.80
Harness,13.31,16.64,19.97
Instrument,73.50,87.62,105.15
OBDH,18.90,20.70,24.83
Propulsion,32.20,33.66,35.99
Structure,95.00,113.10,135.72
Thermal,16.80,19.99,23.99


### By responsibility, and by location

The same rows summed on the other two axes. All three views reconcile to the same total, whatever the flags — they are cuts of one table, not separate calculations.

The harness rows take their location's name as their responsibility, so they are counted here as well as in the location view, while staying their own line under `Harness` in the subsystem view above.

In [10]:
mass_data.by_responsibility()

,mass
responsibility,
Payload,152.38
Platform,318.24


In [11]:
mass_data.by_location()

,mass
location,
Payload,155.41
Platform,315.21
